In [4]:
import pandas as pd
import numpy as np

# =================================================================
# 1. LOAD
# =================================================================
train_test = pd.read_csv("train-test.csv", index_col="load_id")
val = pd.read_csv("validation.csv", index_col="load_id")
chart = pd.read_csv("december-chart-inputs.csv")

In [5]:
# 2. ROW-LEVEL CLEANING + DATE FEATURES (safe before the split)
# Dates stay as plain numbers: cyclical encoding was tested and
# increased test error (less than one year of data)
# =================================================================
for d in (train_test, val, chart):
    d["date"] = pd.to_datetime(d["date"])
    d.loc[d["weight"] <= 0, "weight"] = np.nan
    d["day_of_week"] = d["date"].dt.dayofweek
    d["month"] = d["date"].dt.month

In [6]:
# =================================================================
# 3. LOOKUPS FROM FEATURES ONLY (no target involved)
# =================================================================
# City -> coordinates, for the December file (it has no lat/lon)
both = pd.concat([train_test, val])
coords = pd.concat([
    both[["pickup", "pickup_lat", "pickup_lon"]].set_axis(["city", "lat", "lon"], axis=1),
    both[["delivery", "delivery_lat", "delivery_lon"]].set_axis(["city", "lat", "lon"], axis=1),
]).drop_duplicates("city").set_index("city")

In [7]:
# Date -> median market index, for filling missing values
# (validation dates are filled from other validation loads on the same day)
daily_index = both.groupby("date")["market_index"].median()

In [8]:
# December file: add coordinates and market index
for side in ("pickup", "delivery"):
    chart[f"{side}_lat"] = chart[side].map(coords["lat"])
    chart[f"{side}_lon"] = chart[side].map(coords["lon"])
chart["market_index"] = chart["date"].map(daily_index)


In [9]:
# =================================================================
# 4. TIME-BASED SPLIT
# =================================================================
train_test = train_test.sort_values("date")
cutoff = train_test["date"].quantile(0.8)
train = train_test[train_test["date"] < cutoff].copy()
test = train_test[train_test["date"] >= cutoff].copy()
print("split cutoff:", cutoff.date(), "| train:", train.shape, "| test:", test.shape)

split cutoff: 2025-08-31 | train: (38330, 15) | test: (9670, 15)


In [10]:
FEATURES = [
    "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon",
    "distance", "weight", "market_index", "quote_signal",
    "day_of_week", "month",
    "equipment_Dry Van", "equipment_Flatbed", "equipment_Reefer",
]
TARGET = "posted_rate"
BANDS = [0, 300, 800, 1500, 4000]

In [11]:
# 5. OUTLIERS: find the gaps and drop out-of-bounds loads (training data only)
def remove_outliers(d):
    d = d.copy()
    d["rpm"] = d[TARGET] / d["distance"]
    d["dist_band"] = pd.cut(d["distance"], BANDS)

    group = d.groupby(["equipment", "dist_band"], observed=True)["rpm"]
    typical = group.median()
    ratio = d["rpm"] / group.transform("median")

    s = ratio.sort_values().reset_index(drop=True)
    jumps = s.diff()
    i = jumps[s <= 1].idxmax()
    j = jumps[s.shift(1) >= 1].idxmax()
    low = (s[i - 1] + s[i]) / 2
    high = (s[j - 1] + s[j]) / 2
    print(f"gaps: {s[i-1]:.3f}->{s[i]:.3f} and {s[j-1]:.3f}->{s[j]:.3f} | cutoffs {low:.3f}, {high:.3f}")

    bounds = pd.DataFrame({"min_rpm": typical * low, "max_rpm": typical * high})
    d = d.join(bounds, on=["equipment", "dist_band"])
    keep = d["rpm"].between(d["min_rpm"], d["max_rpm"])
    print(f"outliers dropped: {(~keep).sum()} of {len(d)}")
    return d[keep].drop(columns=["rpm", "dist_band", "min_rpm", "max_rpm"])

In [12]:
# 6. FILL VALUES: learned from training data only
def fit_fill_values(d):
    return {
        "weight_medians": d.groupby("equipment")["weight"].median(),
        "quote_median": d["quote_signal"].median(),
    }


In [13]:
# 7. APPLY FILLS + ENCODE: works on any file (no rows are ever dropped here)
def prepare(d, fill):
    d = d.copy()
    d["weight"] = d["weight"].fillna(d["equipment"].map(fill["weight_medians"]))
    d["market_index"] = d["market_index"].fillna(d["date"].map(daily_index))
    if "quote_signal" not in d:
        d["quote_signal"] = np.nan
    d["quote_signal"] = d["quote_signal"].fillna(fill["quote_median"])
    d = pd.get_dummies(d, columns=["equipment"], dtype=int)
    return d.reindex(columns=FEATURES, fill_value=0)

In [14]:
# STAGE 1: train on train, evaluate on test
# =================================================================
train = remove_outliers(train)
fill = fit_fill_values(train)

X_train, y_train = prepare(train, fill), train[TARGET]
X_test, y_test = prepare(test, fill), test[TARGET]            # test keeps its outliers

# =================================================================
# STAGE 2: all labeled data for the final model
# =================================================================
full = remove_outliers(train_test)
final_fill = fit_fill_values(full)

X_full, y_full = prepare(full, final_fill), full[TARGET]
X_val = prepare(val, final_fill)                              # all 12,000 rows, index = load_id
X_chart = prepare(chart, final_fill)

# =================================================================
# CHECKS
# =================================================================
for name, X in [("train", X_train), ("test", X_test), ("full", X_full),
                ("validation", X_val), ("chart", X_chart)]:
    assert X.isnull().sum().sum() == 0, f"nulls in {name}"
    print(f"{name:11s} {X.shape}")

gaps: 0.467->0.811 and 1.343->2.141 | cutoffs 0.639, 1.742
outliers dropped: 532 of 38330
gaps: 0.466->0.807 and 1.338->2.140 | cutoffs 0.637, 1.739
outliers dropped: 677 of 48000
train       (37798, 13)
test        (9670, 13)
full        (47323, 13)
validation  (12000, 13)
chart       (31, 13)
